In [1]:
import torch
import pandas as pd
from transformers import Trainer, TrainingArguments, AutoModelForSeq2SeqLM, AutoTokenizer
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split


d:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# ✅ Enable CUDA optimizations for RTX 4050
torch.backends.cuda.matmul.allow_tf32 = True  

# ✅ Load CrisisFACTS dataset
df = pd.read_csv("disaster_info_with_embeddings.csv")  # (17325, 17)

# ✅ Use `combined` column for input & set dummy target
df["input_text"] = df["combined"]
df["target_text"] = "Summary: "  # Placeholder text for fine-tuning

# ✅ Split dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["input_text"], df["target_text"], test_size=0.2, random_state=42
)

print("✅ Dataset prepared!")


✅ Dataset prepared!


In [ ]:
#51016

In [3]:
# ✅ Load tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenize data (Batch-wise to avoid memory issues)
def tokenize_function(texts):
    return tokenizer(
        list(texts), padding="longest", truncation=True, max_length=512, return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

train_labels = tokenize_function(train_labels)
val_labels = tokenize_function(val_labels)

print("✅ Tokenization complete!")

✅ Tokenization complete!


In [4]:
# ✅ Custom Dataset Class
class DisasterDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels["input_ids"][idx],
        }


# ✅ Create datasets
train_dataset = DisasterDataset(train_encodings, train_labels)
val_dataset = DisasterDataset(val_encodings, val_labels)

print("✅ Dataset ready for training!")

✅ Dataset ready for training!


In [5]:
# ✅ Load 8-bit Optimized Model
from transformers import BitsAndBytesConfig
from optimum.bettertransformer import BetterTransformer
import deepspeed
import numpy as np

# ✅ Enable quantization (if using bitsandbytes)
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,  # Enable 8-bit quantization
    llm_int8_threshold=6.0,
)

# ✅ Set device to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load model with automatic GPU placement (Fix for Meta Tensor error)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn",
    # quantization_config=quantization_config,  # Ensure quantization is enabled
    #device_map="auto"  # Automatically place model on GPU
).to(device, non_blocking=True)

print("✅ Model successfully converted to BetterTransformer and loaded on GPU!")

# ✅ Enable gradient checkpointing & FA2
# try:
#     model = BetterTransformer.transform(model)  # ⚡ Faster inference with FA2
#     print("✅ Flash Attention 2 enabled!")
# except ImportError:
#     print("⚠️ Optimum not installed, skipping FA2 optimization.")

model.gradient_checkpointing_enable()  # ✅ Saves memory


# ✅ Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,  
    deepspeed="ds_config.json"  # ✅ Pass DeepSpeed config
)


# ✅ Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ✅ Train the model
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

[2025-03-11 13:42:02,942] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


W0311 13:42:03.686000 11860 torch\distributed\elastic\multiprocessing\redirects.py:27] NOTE: Redirects are currently not supported in Windows or MacOs.


✅ Model successfully converted to BetterTransformer and loaded on GPU!
🚀 Starting training...
Parameter Offload: Total persistent parameters: 397312 in 316 params


d:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\transformers\models\bart\modeling_bart.py:496: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
500,0.251900


 stage3_gather_16bit_weights_on_model_save=false. Saving the full checkpoint instead, use zero_to_fp32.py to recover weights
d:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\transformers\modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
 stage3_gather_16bit_weights_on_model_save=false. Saving the full checkpoint instead, use zero_to_fp32.py to recover weights


✅ Training complete!


In [6]:
model.save_pretrained("fine_tuned_disaster_summarizer")
tokenizer.save_pretrained("fine_tuned_disaster_summarizer")

print("✅ Model saved successfully as 'fine_tuned_disaster_summarizer'!")


✅ Model saved successfully as 'fine_tuned_disaster_summarizer'!


In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# ✅ Load the dataset
df = pd.read_csv("disaster_info_with_embeddings.csv")

# ✅ Ensure required columns exist
required_columns = ["Event Name", "combined"]
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing columns in dataset: {missing_cols}")

# ✅ Load fine-tuned model with proper error handling
try:
    model = AutoModelForSeq2SeqLM.from_pretrained("fine_tuned_disaster_summarizer")
    tokenizer = AutoTokenizer.from_pretrained("fine_tuned_disaster_summarizer")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    exit()

def retrieve_disaster_info(disaster_name):
    """Retrieve disaster details based on Event Name."""
    return df[df["Event Name"].str.contains(disaster_name, case=False, na=False)]

def generate_summary(disaster_name):
    """Generate a structured summary using the 'combined' feature."""
    disaster_info = retrieve_disaster_info(disaster_name)
    
    if disaster_info.empty:
        return f"🤖: Sorry, I couldn't find any information on '{disaster_name}'. Try another event?"

    details = " ".join(disaster_info["combined"].dropna().tolist())
    
    # ✅ Truncate input to avoid exceeding model limits
    inputs = tokenizer(details, return_tensors="pt", truncation=True, max_length=512).to(device)

    # ✅ Generate summary
    with torch.no_grad():
        summary_ids = model.generate(inputs.input_ids, max_length=150, num_beams=5, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return f"🤖: Here's a summary of **{disaster_name}**:- {summary}"

# 🔥 Chatbot Loop
while True:
    disaster_name = input("You: ").strip()
    if disaster_name.lower() in ["exit", "quit", "bye"]:
        print("🤖: Goodbye! Stay safe! 🌍")
        break
    print(generate_summary(disaster_name))

d:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


❌ Model loading failed: Error(s) in loading state_dict for BartForConditionalGeneration:
	size mismatch for model.shared.weight: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([50264, 1024]).
	size mismatch for model.encoder.embed_tokens.weight: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([50264, 1024]).
	size mismatch for model.encoder.embed_positions.weight: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([1026, 1024]).
	size mismatch for model.encoder.layers.0.self_attn.k_proj.weight: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([1024, 1024]).
	size mismatch for model.encoder.layers.0.self_attn.v_proj.weight: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([1024, 1024]).
	size mismatch for model.encoder.layers.0.se

: 